# Lab 5. Difference Equations and Dynamic Behavior

**Course:** MATH 7339 Machine Learning and Statistical Learning Theory 2  
**Book:** Modern Time Series Analysis and Sequence Learning  
**Open in Google Colab:** [Lab 5 Colab link](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-05-difference-equations-dynamic-behavior-lab.ipynb)

This lab is designed for **independent study**. It includes the background ideas before programming, then guides you through simulations and interpretation.

## What you will learn

By the end of this lab, you should be able to:

1. Simulate deterministic linear difference equations.
2. Explain why roots determine decay, oscillation, and explosion.
3. Connect recurrence roots to AR model stationarity.
4. Compute impulse-response weights for AR models.
5. Interpret AR(2) behavior using roots, companion matrices, and plots.
6. Use AI tools to check reasoning without replacing mathematical verification.

## Connection with Chapter 5

Chapter 5 studies how a time series model can be understood through a linear recurrence. In the ARMA notes, AR models are written using the backshift operator, and homogeneous linear difference equations are solved by looking at the roots of a characteristic polynomial. The same root idea explains whether an AR process is stable, oscillatory, or explosive.

We will use only elementary Python tools: `numpy`, `pandas`, and `matplotlib`.

> **Colab note:** This notebook avoids the LaTeX command for calligraphic letters because it often displays poorly in Google Colab markdown.

## 0. Mathematical background before programming

### 0.1 First-order difference equation

A first-order homogeneous linear difference equation has the form

$$
x_t = a x_{t-1}.
$$

If the initial value is $x_0$, then repeated substitution gives

$$
x_t = a^t x_0.
$$

Therefore:

- if $|a| < 1$, then $x_t \to 0$;
- if $|a| = 1$, then the sequence does not decay;
- if $|a| > 1$, then the magnitude grows;
- if $a < 0$, the signs alternate.

This simple example already contains the main idea: **roots control dynamics**.

### 0.2 Second-order difference equation

A second-order homogeneous linear difference equation has the form

$$
x_t = \phi_1 x_{t-1} + \phi_2 x_{t-2}.
$$

Try a solution of the form $x_t = \lambda^t$. Substituting gives

$$
\lambda^t = \phi_1 \lambda^{t-1} + \phi_2 \lambda^{t-2}.
$$

After dividing by $\lambda^{t-2}$, we get the characteristic equation

$$
\lambda^2 - \phi_1 \lambda - \phi_2 = 0.
$$

The two roots $\lambda_1, \lambda_2$ determine the solution shape.

- Real positive roots give monotone growth or decay.
- Real negative roots give alternating behavior.
- Complex roots give oscillations.
- Magnitudes below 1 give damping.
- Magnitudes above 1 give explosion.

### 0.3 Connection with AR(2)

An AR(2) process has the form

$$
X_t = \phi_1 X_{t-1} + \phi_2 X_{t-2} + W_t,
$$

where $W_t$ is white noise. The deterministic part is exactly the recurrence above.

The AR polynomial is

$$
1 - \phi_1 z - \phi_2 z^2.
$$

The AR(2) process is stationary and causal when the roots of this AR polynomial lie outside the unit circle. These AR roots are reciprocals of the recurrence roots $\lambda$. Thus, stability of the recurrence is equivalent to the condition that all $|\lambda_i| < 1$.

### 0.4 Companion matrix

The recurrence can also be written as a matrix system:

$$
\begin{bmatrix}
x_t \\
x_{t-1}
\end{bmatrix}
=
\begin{bmatrix}
\phi_1 & \phi_2 \\
1 & 0
\end{bmatrix}
\begin{bmatrix}
x_{t-1} \\
x_{t-2}
\end{bmatrix}.
$$

The eigenvalues of this companion matrix are the recurrence roots $\lambda_1,\lambda_2$.

In [ ]:
# Basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
rng = np.random.default_rng(7339)

# Display settings
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True

## 1. First-order dynamics

We begin with

$$
x_t = a x_{t-1}.
$$

The exact solution is $x_t = a^t x_0$.

In the next cell, we simulate several values of $a$ and compare their behavior.

In [ ]:
def simulate_first_order(a, x0=1.0, n=40):
    # Simulate x_t = a x_{t-1}.
    x = np.zeros(n)
    x[0] = x0
    for t in range(1, n):
        x[t] = a * x[t - 1]
    return x

n = 40
a_values = [0.5, -0.5, 1.0, -1.0, 1.1, -1.1]

plt.figure(figsize=(10, 5))
for a in a_values:
    x = simulate_first_order(a, x0=1.0, n=n)
    plt.plot(x, marker="o", markersize=3, label=f"a={a}")

plt.axhline(0, linewidth=1)
plt.title("First-order difference equation: $x_t = a x_{t-1}$")
plt.xlabel("t")
plt.ylabel("x_t")
plt.legend()
plt.show()

### Checkpoint 1

Answer these questions before moving on:

1. Which values of $a$ lead to decay?
2. Which values lead to sign alternation?
3. Which values lead to explosive behavior?
4. Why does $a=-1$ not converge to zero even though it stays bounded?

In [ ]:
summary_first_order = []
for a in a_values:
    behavior = []
    if abs(a) < 1:
        behavior.append("decays")
    elif abs(a) == 1:
        behavior.append("bounded but not decaying")
    else:
        behavior.append("explodes")
    if a < 0:
        behavior.append("alternates signs")
    else:
        behavior.append("no sign alternation from a")
    summary_first_order.append({"a": a, "|a|": abs(a), "behavior": "; ".join(behavior)})

pd.DataFrame(summary_first_order)

## 2. Second-order dynamics and characteristic roots

Now consider

$$
x_t = \phi_1 x_{t-1} + \phi_2 x_{t-2}.
$$

The characteristic equation is

$$
\lambda^2 - \phi_1 \lambda - \phi_2 = 0.
$$

The function below simulates the recurrence and computes its roots.

In [ ]:
def simulate_second_order(phi1, phi2, x0=1.0, x1=0.5, n=80):
    # Simulate x_t = phi1*x_{t-1} + phi2*x_{t-2}.
    x = np.zeros(n)
    x[0] = x0
    x[1] = x1
    for t in range(2, n):
        x[t] = phi1 * x[t - 1] + phi2 * x[t - 2]
    return x


def recurrence_roots(phi1, phi2):
    # Roots of lambda^2 - phi1*lambda - phi2 = 0.
    return np.roots([1.0, -phi1, -phi2])


def companion_matrix(phi1, phi2):
    return np.array([[phi1, phi2], [1.0, 0.0]])

examples = {
    "damped positive roots": (0.7, -0.1),
    "alternating root effect": (-0.5, 0.2),
    "damped oscillation": (1.2, -0.8),
    "explosive": (1.3, 0.2),
}

rows = []
for name, (phi1, phi2) in examples.items():
    roots = recurrence_roots(phi1, phi2)
    eigvals = np.linalg.eigvals(companion_matrix(phi1, phi2))
    rows.append({
        "case": name,
        "phi1": phi1,
        "phi2": phi2,
        "root 1": roots[0],
        "root 2": roots[1],
        "max |root|": np.max(np.abs(roots)),
        "stable deterministic part?": np.max(np.abs(roots)) < 1,
        "companion eigenvalues match roots?": np.allclose(np.sort_complex(roots), np.sort_complex(eigvals))
    })

pd.DataFrame(rows)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
axes = axes.ravel()

for ax, (name, (phi1, phi2)) in zip(axes, examples.items()):
    x = simulate_second_order(phi1, phi2, x0=1.0, x1=0.5, n=80)
    roots = recurrence_roots(phi1, phi2)
    ax.plot(x, linewidth=2)
    ax.axhline(0, linewidth=1)
    ax.set_title(f"{name}\nphi1={phi1}, phi2={phi2}, max |root|={np.max(np.abs(roots)):.3f}")
    ax.set_xlabel("t")
    ax.set_ylabel("x_t")

plt.tight_layout()
plt.show()

### Checkpoint 2

The plots above illustrate four different behaviors.

For each case, explain:

1. Are the roots real or complex?
2. Is the largest root magnitude less than 1?
3. Does the sequence decay, oscillate, or explode?
4. How does the plot confirm the root calculation?

## 3. Complex roots and damped oscillations

Complex roots are easiest to understand in polar form:

$$
\lambda_{1,2} = r e^{\pm i\omega}.
$$

The general real-valued solution behaves like a damped sinusoid:

$$
x_t \approx C r^t \cos(\omega t + \alpha).
$$

The number $r$ controls damping or explosion. The angle $\omega$ controls the oscillation frequency.

For a second-order recurrence with complex-conjugate roots $r e^{\pm i\omega}$, the coefficients are

$$
\phi_1 = 2r\cos(\omega), \qquad \phi_2 = -r^2.
$$

In [ ]:
def ar2_coefficients_from_complex_roots(r, omega):
    # Return phi1, phi2 for recurrence roots r*exp(+/- i omega).
    phi1 = 2 * r * np.cos(omega)
    phi2 = -r**2
    return phi1, phi2

omega = np.pi / 6
r_values = [0.8, 1.0, 1.05]

plt.figure(figsize=(10, 5))
for r in r_values:
    phi1, phi2 = ar2_coefficients_from_complex_roots(r, omega)
    x = simulate_second_order(phi1, phi2, x0=1.0, x1=np.cos(omega), n=100)
    plt.plot(x, label=f"r={r}, phi1={phi1:.3f}, phi2={phi2:.3f}")

plt.axhline(0, linewidth=1)
plt.title("Complex roots: damping, persistence, and explosion")
plt.xlabel("t")
plt.ylabel("x_t")
plt.legend()
plt.show()

### Checkpoint 3

In the previous plot, the angle $\omega$ is the same for all three curves, but the magnitude $r$ changes.

Explain why the frequency of oscillation is similar, while the long-run amplitude is different.

## 4. AR polynomial roots versus recurrence roots

For AR(2), the AR polynomial is

$$
1 - \phi_1 z - \phi_2 z^2.
$$

The stationarity condition is that the roots in $z$ lie outside the unit circle.

The recurrence roots $\lambda$ solve

$$
\lambda^2 - \phi_1 \lambda - \phi_2 = 0.
$$

These are related by

$$
z = \frac{1}{\lambda}.
$$

So the two equivalent stability statements are:

- recurrence roots satisfy $|\lambda_i| < 1$;
- AR polynomial roots satisfy $|z_i| > 1$.

In [ ]:
def ar_polynomial_roots(phi1, phi2):
    # Roots of 1 - phi1*z - phi2*z^2 = 0.
    return np.roots([-phi2, -phi1, 1.0])

rows = []
for name, (phi1, phi2) in examples.items():
    lam = recurrence_roots(phi1, phi2)
    z_roots = ar_polynomial_roots(phi1, phi2)
    rows.append({
        "case": name,
        "max |lambda|": np.max(np.abs(lam)),
        "min |z root|": np.min(np.abs(z_roots)),
        "stable by lambda?": np.max(np.abs(lam)) < 1,
        "stationary by AR roots?": np.min(np.abs(z_roots)) > 1,
    })

pd.DataFrame(rows)

In [ ]:
def plot_ar_roots(phi1, phi2, title="AR polynomial roots"):
    roots = ar_polynomial_roots(phi1, phi2)
    angle = np.linspace(0, 2*np.pi, 400)
    unit_x = np.cos(angle)
    unit_y = np.sin(angle)

    plt.figure(figsize=(5.5, 5.5))
    plt.plot(unit_x, unit_y, linestyle="--", label="unit circle")
    plt.scatter(np.real(roots), np.imag(roots), s=80, label="AR roots")
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.title(title)
    plt.xlabel("Real part")
    plt.ylabel("Imaginary part")
    plt.legend()
    plt.show()

plot_ar_roots(1.2, -0.8, "AR roots for damped oscillation example")

### Checkpoint 4

For the root plot above:

1. Are the AR roots inside or outside the unit circle?
2. What does this imply about stationarity?
3. How is this related to the damped oscillation you saw earlier?

## 5. Impulse-response weights for AR models

For a causal AR model, we can write

$$
X_t = \sum_{j=0}^{\infty} \psi_j W_{t-j}.
$$

The coefficients $\psi_j$ are called impulse-response weights. They tell us how a shock today affects future values.

For AR(2), the recursion is

$$
\psi_0 = 1,
$$

$$
\psi_j = \phi_1 \psi_{j-1} + \phi_2 \psi_{j-2}, \qquad j \ge 1,
$$

where we set $\psi_j=0$ for $j<0$.

In [ ]:
def impulse_response_ar(phi, n_terms=40):
    # Compute impulse-response weights for AR(p).
    # phi contains AR coefficients [phi1, ..., phip].
    # n_terms is the number of psi weights to compute.
    phi = np.asarray(phi, dtype=float)
    p = len(phi)
    psi = np.zeros(n_terms)
    psi[0] = 1.0
    for j in range(1, n_terms):
        total = 0.0
        for k in range(1, p + 1):
            if j - k >= 0:
                total += phi[k - 1] * psi[j - k]
        psi[j] = total
    return psi

ar_cases = {
    "AR(1), phi=0.7": [0.7],
    "AR(1), phi=-0.7": [-0.7],
    "AR(2), damped oscillation": [1.2, -0.8],
    "AR(2), explosive": [1.3, 0.2],
}

plt.figure(figsize=(10, 5))
for name, phi in ar_cases.items():
    psi = impulse_response_ar(phi, n_terms=40)
    plt.plot(psi, marker="o", markersize=3, label=name)

plt.axhline(0, linewidth=1)
plt.title("Impulse-response weights for AR models")
plt.xlabel("lag j")
plt.ylabel("psi_j")
plt.legend()
plt.show()

### Interpretation

The impulse-response plot tells us how long a shock persists.

- If the weights decay quickly, shocks disappear quickly.
- If the weights oscillate and decay, the model has damped cyclic behavior.
- If the weights grow, the model is not stable.

This is why root conditions are not just abstract algebra: they determine how shocks move through time.

## 6. Simulating AR(2) processes from impulse-response intuition

Now we simulate

$$
X_t = \phi_1 X_{t-1} + \phi_2 X_{t-2} + W_t.
$$

The deterministic roots determine how each random shock propagates.

In [ ]:
def simulate_ar2(phi1, phi2, n=300, burnin=200, sigma=1.0, seed=0):
    local_rng = np.random.default_rng(seed)
    total = n + burnin
    w = local_rng.normal(0, sigma, size=total)
    x = np.zeros(total)
    for t in range(2, total):
        x[t] = phi1 * x[t - 1] + phi2 * x[t - 2] + w[t]
    return x[burnin:]

stable_phi = (1.2, -0.8)
explosive_phi = (1.3, 0.2)

x_stable = simulate_ar2(*stable_phi, n=300, seed=10)
x_explosive = simulate_ar2(*explosive_phi, n=80, burnin=0, seed=10)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(x_stable)
axes[0].set_title("Stable AR(2): damped oscillatory behavior")
axes[0].set_xlabel("t")
axes[0].set_ylabel("X_t")

axes[1].plot(x_explosive)
axes[1].set_title("Explosive AR(2): unstable behavior")
axes[1].set_xlabel("t")
axes[1].set_ylabel("X_t")

plt.tight_layout()
plt.show()

### Important practical warning

An explosive AR process can generate extremely large values quickly. In real modeling, explosive fitted dynamics are usually a warning sign unless there is a special scientific reason for such behavior.

## 7. Sample ACF from scratch

We now compute the sample autocorrelation function to see how the dynamic behavior appears in dependence.

For a series $x_0,\ldots,x_{n-1}$, define the sample autocovariance at lag $h$ by

$$
\hat{\gamma}(h) = \frac{1}{n} \sum_{t=h}^{n-1} (x_t - \bar{x})(x_{t-h}-\bar{x}).
$$

The sample ACF is

$$
\hat{\rho}(h) = \frac{\hat{\gamma}(h)}{\hat{\gamma}(0)}.
$$

In [ ]:
def sample_acf(x, max_lag=40):
    x = np.asarray(x, dtype=float)
    n = len(x)
    x_centered = x - x.mean()
    gamma0 = np.sum(x_centered * x_centered) / n
    acf = []
    for h in range(max_lag + 1):
        gamma_h = np.sum(x_centered[h:] * x_centered[:n-h]) / n
        acf.append(gamma_h / gamma0)
    return np.array(acf)

acf_stable = sample_acf(x_stable, max_lag=50)

plt.figure(figsize=(10, 4))
plt.stem(range(len(acf_stable)), acf_stable)
plt.axhline(0, linewidth=1)
plt.axhline(1.96 / np.sqrt(len(x_stable)), linestyle="--", linewidth=1, label="approx. 95% band")
plt.axhline(-1.96 / np.sqrt(len(x_stable)), linestyle="--", linewidth=1)
plt.title("Sample ACF of stable oscillatory AR(2)")
plt.xlabel("lag")
plt.ylabel("sample ACF")
plt.legend()
plt.show()

### Checkpoint 5

The sample ACF should show oscillation and decay.

Explain why this is consistent with the complex roots of the AR(2) model.

## 8. Model identification practice: match plots to roots

In this section, we generate several AR(2) processes. Your task is to connect the visual behavior to root behavior.

In [ ]:
practice_cases = {
    "A": (0.6, 0.1),
    "B": (-0.7, 0.1),
    "C": (1.4, -0.85),
    "D": (0.2, -0.9),
}

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.ravel()

root_table = []
for ax, (label, (phi1, phi2)) in zip(axes, practice_cases.items()):
    x = simulate_ar2(phi1, phi2, n=250, seed=100 + ord(label))
    roots_lambda = recurrence_roots(phi1, phi2)
    roots_z = ar_polynomial_roots(phi1, phi2)
    ax.plot(x)
    ax.set_title(f"Case {label}: phi1={phi1}, phi2={phi2}")
    ax.set_xlabel("t")
    ax.set_ylabel("X_t")
    root_table.append({
        "case": label,
        "phi1": phi1,
        "phi2": phi2,
        "recurrence roots": roots_lambda,
        "max |lambda|": np.max(np.abs(roots_lambda)),
        "AR roots": roots_z,
        "min |z root|": np.min(np.abs(roots_z)),
        "stationary?": np.min(np.abs(roots_z)) > 1,
    })

plt.tight_layout()
plt.show()

pd.DataFrame(root_table)

### Practice questions

For each case A-D:

1. Is the process stationary according to the AR root condition?
2. Does the plot look stable?
3. Does the plot show oscillation?
4. Do the recurrence roots explain what you see?

## 9. From deterministic recurrence to stochastic forecasting intuition

For a stable AR process, future forecasts depend on how the deterministic recurrence propagates the current state.

For AR(2), if the most recent values are $X_t$ and $X_{t-1}$, then the one-step forecast is

$$
\hat{X}_{t+1} = \phi_1 X_t + \phi_2 X_{t-1}.
$$

The two-step forecast applies the recurrence again:

$$
\hat{X}_{t+2} = \phi_1 \hat{X}_{t+1} + \phi_2 X_t.
$$

The forecast path follows the deterministic difference equation. This is another reason that roots matter: they describe how forecasts decay back toward the mean or oscillate around it.

In [ ]:
def forecast_ar2_path(phi1, phi2, x_t, x_tm1, horizon=40):
    # Forecast future mean path for AR(2) with zero mean.
    path = np.zeros(horizon)
    prev1 = x_t
    prev2 = x_tm1
    for h in range(horizon):
        next_val = phi1 * prev1 + phi2 * prev2
        path[h] = next_val
        prev2 = prev1
        prev1 = next_val
    return path

phi1, phi2 = stable_phi
x_t = 2.0
x_tm1 = -1.0
forecast_path = forecast_ar2_path(phi1, phi2, x_t, x_tm1, horizon=60)

plt.figure(figsize=(10, 4))
plt.plot(range(-1, 1), [x_tm1, x_t], marker="o", linewidth=2, label="observed state")
plt.plot(range(1, 61), forecast_path, marker="o", markersize=3, label="forecast mean path")
plt.axhline(0, linewidth=1)
plt.title("AR(2) forecast mean path follows the deterministic recurrence")
plt.xlabel("relative time")
plt.ylabel("value")
plt.legend()
plt.show()

### Checkpoint 6

Why does the forecast path eventually return toward zero in this stable example?

How would the forecast path behave if the recurrence roots had magnitude greater than 1?

## 10. Mini-project: design your own AR(2) dynamics

Choose one of the following goals:

1. A stable process with no visible oscillation.
2. A stable process with damped oscillation.
3. An unstable process.
4. A process with alternating signs.

Then choose $\phi_1,\phi_2$, compute the roots, simulate a series, plot the impulse-response weights, and explain the behavior.

In [ ]:
# Mini-project starter code. Modify phi1 and phi2.
phi1 = 1.1
phi2 = -0.6

roots_lambda = recurrence_roots(phi1, phi2)
roots_z = ar_polynomial_roots(phi1, phi2)
print("Recurrence roots lambda:", roots_lambda)
print("Max |lambda|:", np.max(np.abs(roots_lambda)))
print("AR polynomial roots z:", roots_z)
print("Min |z|:", np.min(np.abs(roots_z)))
print("Stationary by AR root condition?", np.min(np.abs(roots_z)) > 1)

x = simulate_ar2(phi1, phi2, n=300, seed=2026)
psi = impulse_response_ar([phi1, phi2], n_terms=50)
acf = sample_acf(x, max_lag=40)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(x)
axes[0].set_title("Simulated AR(2)")
axes[0].set_xlabel("t")

axes[1].plot(psi, marker="o", markersize=3)
axes[1].axhline(0, linewidth=1)
axes[1].set_title("Impulse response")
axes[1].set_xlabel("lag")

axes[2].stem(range(len(acf)), acf)
axes[2].axhline(0, linewidth=1)
axes[2].set_title("Sample ACF")
axes[2].set_xlabel("lag")

plt.tight_layout()
plt.show()

## 11. AI-assisted study prompts

Use these prompts with an AI assistant, but verify the answer using your code and root calculations.

### Prompt 1: Root interpretation

> I have an AR(2) model $X_t = \phi_1 X_{t-1} + \phi_2 X_{t-2} + W_t$. Explain how the recurrence roots and AR polynomial roots are related. Then explain the stationarity condition in both languages.

### Prompt 2: Debugging simulation

> Here is my code for simulating an AR(2) process. Check whether the indexing is correct and whether I have included a burn-in period properly.

### Prompt 3: Plot interpretation

> I simulated an AR(2) process and its ACF oscillates while decaying. What does that suggest about the characteristic roots?

### Prompt 4: Forecast behavior

> Why does the multi-step forecast path of a stable AR(2) process follow a deterministic difference equation?

### Important verification rule

Never accept an AI explanation unless you can verify it using at least one of the following:

1. characteristic roots,
2. AR polynomial roots,
3. companion matrix eigenvalues,
4. impulse-response weights,
5. simulation plots.

## 12. Lab reflection

Write a short response to each question.

1. What is the difference between recurrence roots $\lambda$ and AR polynomial roots $z$?
2. Why do complex roots lead to oscillations?
3. Why does $|\lambda|<1$ imply stable deterministic behavior?
4. Why does the AR root condition require roots outside the unit circle?
5. How do impulse-response weights help explain the effect of a shock?
6. What is one way this lab connects to forecasting?

## 13. Checklist

Before finishing, make sure you can:

- [ ] simulate a first-order difference equation;
- [ ] simulate a second-order difference equation;
- [ ] compute recurrence roots;
- [ ] compute AR polynomial roots;
- [ ] explain the reciprocal relationship between the two root systems;
- [ ] compute impulse-response weights for an AR model;
- [ ] identify damped oscillation from roots and plots;
- [ ] explain why unstable roots create explosive behavior;
- [ ] use AI as a critic, not as a replacement for verification.